In [ ]:
from pathlib import Path

MODEL_PATH = Path("models/checkpoint_epoch_50.pth")

Load model

In [ ]:
import torch
from torchvision.models import efficientnet_b0
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = checkpoint['model']
model_types = checkpoint['all_types']
model = model.to(device)
model.eval()

print(f"Model was trained on {len(model_types)} creature types")

Read data

In [ ]:
import pandas as pd

df = pd.read_csv(f"data/oracle_cards_manifest.csv")
print(f"Total cards: {len(df)}")
print(df.head())
print(df["types"].value_counts().head(20))

Build dataset

In [ ]:
from PIL import Image
from torch.utils.data import Dataset
import torch


class CreatureDataset(Dataset):
    def __init__(self, df, all_types, transform=None):
        self.df = df.reset_index(drop=True)
        self.all_types = all_types
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        
        label = torch.zeros(len(self.all_types))
        types = row["types"].split("|") if isinstance(row["types"], str) else []
        for t in types:
            if t in self.all_types:
                label[self.all_types.index(t)] = 1.0
        
        return image, label

In [ ]:
all_types = sorted(set(
    t for types_str in df["types"] if isinstance(types_str, str) for t in types_str.strip().split("|")
))
print(all_types)
print(f"amount of creature types:", len(all_types))

In [ ]:
model_types_set = set(model_types)

def is_compatible(types_str):
    if not isinstance(types_str, str):
        return True  # typeless cards are fine
    return all(t in model_types_set for t in types_str.split("|"))

df_eval = df[df["types"].apply(is_compatible)].reset_index(drop=True)
print(f"Evaluation cards after filtering: {len(df_eval)}")

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_dataset = CreatureDataset(df_eval, model_types, transform=val_transform)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)

print(f"Evaluation dataset size: {len(eval_dataset)}")

Evaluate

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in eval_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.sigmoid(outputs).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print(f"Predictions shape: {all_preds.shape}")
print(f"Labels shape: {all_labels.shape}")



In [ ]:
# get binary predictions
best_f1 = 0
best_threshold = 0.5
step = 0.01

for i in range(1, 100):
    threshold = i * step
    binary_preds = (all_preds > threshold).float()
    
    tp = ((binary_preds == 1) & (all_labels == 1)).sum().item()
    fp = ((binary_preds == 1) & (all_labels == 0)).sum().item()
    fn = ((binary_preds == 0) & (all_labels == 1)).sum().item()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
        best_binary_preds = binary_preds

print(f"best threshold: {best_threshold:.2f}, best f1: {best_f1:.3f}")
binary_preds = best_binary_preds



In [ ]:
results = []
for i, creature_type in enumerate(model_types):
    tp = ((binary_preds[:, i] == 1) & (all_labels[:, i] == 1)).sum().item()
    fp = ((binary_preds[:, i] == 1) & (all_labels[:, i] == 0)).sum().item()
    fn = ((binary_preds[:, i] == 0) & (all_labels[:, i] == 1)).sum().item()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    results.append({
        'type': creature_type,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'support': int(all_labels[:, i].sum().item())  # how many true examples
    })

results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(all_preds.numpy().flatten(), bins=100)
plt.xlabel("sigmoid output")
plt.ylabel("count")
plt.title("distribution of model outputs")
plt.yscale("log")
plt.show()

In [ ]:
# outputs where the true label is 1 (correct types)
correct_mask = all_labels == 1
incorrect_mask = all_labels == 0

correct_outputs = all_preds[correct_mask].numpy()
incorrect_outputs = all_preds[incorrect_mask].numpy()

plt.figure(figsize=(10, 4))
plt.hist(incorrect_outputs, bins=100, alpha=0.5, label='true negatives', color='blue')
plt.hist(correct_outputs, bins=100, alpha=0.5, label='true positives', color='red')
plt.xlabel("sigmoid output")
plt.ylabel("count")
plt.yscale("log")
plt.legend()
plt.title("model outputs separated by true label")
plt.show()